In [1]:
import json
from pyflink.common import Types, WatermarkStrategy
from pyflink.common.types import Row
from pyflink.common.serialization import SimpleStringSchema
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.connectors.kafka import KafkaSource, KafkaOffsetsInitializer
from pyflink.datastream.connectors.jdbc import JdbcSink, JdbcConnectionOptions, JdbcExecutionOptions

In [2]:
env = StreamExecutionEnvironment.get_execution_environment()

env.add_jars(
    "file:///opt/flink-jars/flink-sql-connector-kafka-3.0.1-1.17.jar",
    "file:///opt/flink-jars/flink-connector-jdbc-3.1.1-1.17.jar",
    "file:///opt/flink-jars/postgresql-42.6.0.jar"
)

In [3]:
fat_type_info = Types.ROW_NAMED(
    ["id", "customer_first_name", "customer_last_name", "customer_age", "customer_email",
     "customer_country", "customer_postal_code", "customer_pet_type", "customer_pet_name",
     "customer_pet_breed", "seller_first_name", "seller_last_name", "seller_email",
     "seller_country", "seller_postal_code", "product_name", "product_category",
     "product_price", "product_quantity", "sale_date", "sale_customer_id",
     "sale_seller_id", "sale_product_id", "sale_quantity", "sale_total_price",
     "store_name", "store_location", "store_city", "store_state", "store_country",
     "store_phone", "store_email", "pet_category", "product_weight", "product_color",
     "product_size", "product_brand", "product_material", "product_description",
     "product_rating", "product_reviews", "product_release_date", "product_expiry_date",
     "supplier_name", "supplier_contact", "supplier_email", "supplier_phone",
     "supplier_address", "supplier_city", "supplier_country"],
    [Types.INT(), Types.STRING(), Types.STRING(), Types.INT(), Types.STRING(),
     Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(),
     Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(),
     Types.STRING(), Types.STRING(), Types.FLOAT(), Types.INT(), Types.STRING(),
     Types.INT(), Types.INT(), Types.INT(), Types.INT(), Types.FLOAT(),
     Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(),
     Types.STRING(), Types.STRING(), Types.STRING(), Types.FLOAT(), Types.STRING(),
     Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.FLOAT(),
     Types.INT(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(),
     Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING()]
)

In [4]:
kafka_source = KafkaSource.builder() \
    .set_bootstrap_servers("kafka:9092") \
    .set_topics("raw_data") \
    .set_group_id("flink_datastream_snowflake") \
    .set_starting_offsets(KafkaOffsetsInitializer.earliest()) \
    .set_bounded(KafkaOffsetsInitializer.latest()) \
    .set_value_only_deserializer(SimpleStringSchema()) \
    .build()

ds = env.from_source(kafka_source, WatermarkStrategy.no_watermarks(), "Kafka Source")

In [5]:
def parse_msg(json_str):
    d = json.loads(json_str)
    return Row(
        id=int(d['id']) if d.get('id') else None,
        customer_first_name=str(d.get('customer_first_name', '')),
        customer_last_name=str(d.get('customer_last_name', '')),
        customer_age=int(d['customer_age']) if d.get('customer_age') else None,
        customer_email=str(d.get('customer_email', '')),
        customer_country=str(d.get('customer_country', '')),
        customer_postal_code=str(d.get('customer_postal_code', '')),
        customer_pet_type=str(d.get('customer_pet_type', '')),
        customer_pet_name=str(d.get('customer_pet_name', '')),
        customer_pet_breed=str(d.get('customer_pet_breed', '')),
        seller_first_name=str(d.get('seller_first_name', '')),
        seller_last_name=str(d.get('seller_last_name', '')),
        seller_email=str(d.get('seller_email', '')),
        seller_country=str(d.get('seller_country', '')),
        seller_postal_code=str(d.get('seller_postal_code', '')),
        product_name=str(d.get('product_name', '')),
        product_category=str(d.get('product_category', '')),
        product_price=float(d['product_price']) if d.get('product_price') else 0.0,
        product_quantity=int(d['product_quantity']) if d.get('product_quantity') else 0,
        sale_date=str(d.get('sale_date', '')),
        sale_customer_id=int(d['sale_customer_id']) if d.get('sale_customer_id') else None,
        sale_seller_id=int(d['sale_seller_id']) if d.get('sale_seller_id') else None,
        sale_product_id=int(d['sale_product_id']) if d.get('sale_product_id') else None,
        sale_quantity=int(d['sale_quantity']) if d.get('sale_quantity') else 0,
        sale_total_price=float(d['sale_total_price']) if d.get('sale_total_price') else 0.0,
        store_name=str(d.get('store_name', '')),
        store_location=str(d.get('store_location', '')),
        store_city=str(d.get('store_city', '')),
        store_state=str(d.get('store_state', '')),
        store_country=str(d.get('store_country', '')),
        store_phone=str(d.get('store_phone', '')),
        store_email=str(d.get('store_email', '')),
        pet_category=str(d.get('pet_category', '')),
        product_weight=float(d['product_weight']) if d.get('product_weight') else 0.0,
        product_color=str(d.get('product_color', '')),
        product_size=str(d.get('product_size', '')),
        product_brand=str(d.get('product_brand', '')),
        product_material=str(d.get('product_material', '')),
        product_description=str(d.get('product_description', '')),
        product_rating=float(d['product_rating']) if d.get('product_rating') else 0.0,
        product_reviews=int(d['product_reviews']) if d.get('product_reviews') else 0,
        product_release_date=str(d.get('product_release_date', '')),
        product_expiry_date=str(d.get('product_expiry_date', '')),
        supplier_name=str(d.get('supplier_name', '')),
        supplier_contact=str(d.get('supplier_contact', '')),
        supplier_email=str(d.get('supplier_email', '')),
        supplier_phone=str(d.get('supplier_phone', '')),
        supplier_address=str(d.get('supplier_address', '')),
        supplier_city=str(d.get('supplier_city', '')),
        supplier_country=str(d.get('supplier_country', ''))
    )

parsed_stream = ds.map(parse_msg, output_type=fat_type_info)

In [6]:
def create_jdbc_sink(sql, type_info):
    return JdbcSink.sink(
        sql,
        type_info,
        JdbcConnectionOptions.JdbcConnectionOptionsBuilder()
        .with_url("jdbc:postgresql://postgres:5432/postgres")
        .with_driver_name("org.postgresql.Driver")
        .with_user_name("postgres")
        .with_password("mysecretpassword")
        .build(),
        JdbcExecutionOptions.builder()
        .with_batch_interval_ms(1000)
        .with_batch_size(50)
        .with_max_retries(5)
        .build()
    )

In [7]:
def extract_countries(row):
    countries = {row.customer_country, row.seller_country, row.store_country, row.supplier_country}
    for c in countries:
        if c: yield Row(name=c)

In [8]:
type_dim_countries = Types.ROW_NAMED(['name'], [Types.STRING()])
parsed_stream.flat_map(extract_countries, output_type=type_dim_countries).add_sink(
    create_jdbc_sink("INSERT INTO public.dim_countries (name) VALUES (?) ON CONFLICT (name) DO NOTHING",
                     type_dim_countries)
)

In [9]:
type_dim_pet_types = Types.ROW_NAMED(['pet_type'], [Types.STRING()])
parsed_stream.map(lambda r: Row(pet_type=r.customer_pet_type), output_type=type_dim_pet_types).add_sink(
    create_jdbc_sink("INSERT INTO public.dim_pet_types (pet_type) VALUES (?) ON CONFLICT (pet_type) DO NOTHING",
                     type_dim_pet_types)
)

In [10]:
type_dim_product_categories = Types.ROW_NAMED(['category'], [Types.STRING()])
parsed_stream.map(lambda r: Row(category=r.product_category), output_type=type_dim_product_categories).add_sink(
    create_jdbc_sink(
        "INSERT INTO public.dim_product_categories (category) VALUES (?) ON CONFLICT (category) DO NOTHING",
        type_dim_product_categories)
)

In [11]:
type_dim_pet_categories = Types.ROW_NAMED(['category'], [Types.STRING()])
parsed_stream.map(lambda r: Row(category=r.pet_category), output_type=type_dim_pet_categories).add_sink(
    create_jdbc_sink(
        "INSERT INTO public.dim_pet_categories (category) VALUES (?) ON CONFLICT (category) DO NOTHING",
        type_dim_pet_categories)
)

In [12]:
type_dim_customers = Types.ROW_NAMED(
    ['first_name', 'last_name', 'age', 'email', 'country_name', 'postal_code', 'pet_type_name', 'pet_name',
     'pet_breed'],
    [Types.STRING(), Types.STRING(), Types.INT(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(),
     Types.STRING(), Types.STRING()]
)
parsed_stream.map(
    lambda r: Row(r.customer_first_name, r.customer_last_name, r.customer_age, r.customer_email, r.customer_country,
                  r.customer_postal_code, r.customer_pet_type, r.customer_pet_name, r.customer_pet_breed),
    output_type=type_dim_customers).add_sink(
    create_jdbc_sink("""
                     INSERT INTO public.dim_customers (first_name, last_name, age, email, country_id, postal_code,
                                                       pet_type_id, pet_name, pet_breed)
                     VALUES (?, ?, ?, ?, (SELECT id FROM public.dim_countries WHERE name = ?), ?,
                             (SELECT id FROM public.dim_pet_types WHERE pet_type = ?), ?, ?)
                     """, type_dim_customers)
)

In [13]:
type_dim_sellers = Types.ROW_NAMED(
    ['first_name', 'last_name', 'email', 'country_name', 'postal_code'],
    [Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING()]
)
parsed_stream.map(
    lambda r: Row(r.seller_first_name, r.seller_last_name, r.seller_email, r.seller_country, r.seller_postal_code),
    output_type=type_dim_sellers).add_sink(
    create_jdbc_sink("""
                     INSERT INTO public.dim_sellers (first_name, last_name, email, country_id, postal_code)
                     VALUES (?, ?, ?, (SELECT id FROM public.dim_countries WHERE name = ?), ?)
                     """, type_dim_sellers)
)

In [14]:
type_dim_products = Types.ROW_NAMED(
    ['name', 'category_name', 'price', 'quantity', 'pet_category_name', 'weight', 'color', 'size', 'brand',
     'material', 'description', 'rating', 'reviews', 'release_date', 'expiry_date'],
    [Types.STRING(), Types.STRING(), Types.FLOAT(), Types.INT(), Types.STRING(), Types.FLOAT(), Types.STRING(),
     Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.FLOAT(), Types.INT(), Types.STRING(),
     Types.STRING()]
)
parsed_stream.map(
    lambda r: Row(r.product_name, r.product_category, r.product_price, r.product_quantity, r.pet_category,
                  r.product_weight, r.product_color, r.product_size, r.product_brand, r.product_material,
                  r.product_description, r.product_rating, r.product_reviews, r.product_release_date,
                  r.product_expiry_date), output_type=type_dim_products).add_sink(
    create_jdbc_sink("""
                     INSERT INTO public.dim_products (name, category_id, price, quantity, pet_category_id, weight,
                                                      color, size, brand, material, description, rating, reviews,
                                                      release_date, expiry_date)
                     VALUES (?, (SELECT id FROM public.dim_product_categories WHERE category = ?), ?, ?,
                             (SELECT id FROM public.dim_pet_categories WHERE category = ?), ?, ?, ?, ?, ?, ?, ?, ?,
                             ?, ?)
                     """, type_dim_products)
)

In [15]:
type_dim_stores = Types.ROW_NAMED(
    ['name', 'location', 'city', 'state', 'country_name', 'phone', 'email'],
    [Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING()]
)
parsed_stream.map(
    lambda r: Row(r.store_name, r.store_location, r.store_city, r.store_state, r.store_country, r.store_phone,
                  r.store_email), output_type=type_dim_stores).add_sink(
    create_jdbc_sink("""
                     INSERT INTO public.dim_stores (name, location, city, state, country_id, phone, email)
                     VALUES (?, ?, ?, ?, (SELECT id FROM public.dim_countries WHERE name = ?), ?, ?)
                     """, type_dim_stores)
)

In [16]:
type_dim_suppliers = Types.ROW_NAMED(
    ['name', 'contact', 'email', 'phone', 'address', 'city', 'country_name'],
    [Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING()]
)
parsed_stream.map(
    lambda r: Row(r.supplier_name, r.supplier_contact, r.supplier_email, r.supplier_phone, r.supplier_address,
                  r.supplier_city, r.supplier_country), output_type=type_dim_suppliers).add_sink(
    create_jdbc_sink("""
                     INSERT INTO public.dim_suppliers (name, contact, email, phone, address, city, country_id)
                     VALUES (?, ?, ?, ?, ?, ?, (SELECT id FROM public.dim_countries WHERE name = ?))
                     """, type_dim_suppliers)
)

In [17]:
type_fact_sales = Types.ROW_NAMED(
    ['date', 'customer_email', 'seller_email', 'product_name', 'product_brand', 'product_material',
     'product_description', 'store_email', 'supplier_contact', 'quantity', 'total_price'],
    [Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(),
     Types.STRING(), Types.STRING(), Types.INT(), Types.FLOAT()]
)
parsed_stream.map(lambda r: Row(r.sale_date, r.customer_email, r.seller_email, r.product_name, r.product_brand,
                                r.product_material, r.product_description, r.store_email, r.supplier_contact,
                                r.sale_quantity, r.sale_total_price), output_type=type_fact_sales).add_sink(
    create_jdbc_sink("""
                     INSERT INTO public.fact_sales (date, customer_id, seller_id, product_id, store_id, supplier_id,
                                                    quantity, total_price)
                     VALUES (?,
                             (SELECT id FROM public.dim_customers WHERE email = ? LIMIT 1),
                            (SELECT id FROM public.dim_sellers WHERE email = ? LIMIT 1),
                            (SELECT id FROM public.dim_products WHERE name = ? AND brand = ? AND material = ? AND description = ? LIMIT 1),
                            (SELECT id FROM public.dim_stores WHERE email = ? LIMIT 1),
                            (SELECT id FROM public.dim_suppliers WHERE contact = ? LIMIT 1), ?, ? )
                     """, type_fact_sales)
)

In [18]:
env.execute()